Código para testar as funções do script agent_lizards_1.ipynb

In [17]:
import numpy as np
import pandas as pd
import os
import time
from tqdm import tqdm, trange
import sys
import matplotlib.pyplot as plt

# variaveis
L = 100 # lado do lattice
n_lagartos = L**2 # lagartos que cabem no lattice
estrategias = ['O', 'Y', 'B'] # estratégias possíveis
a = 2 # ganho em fitness ao vencer
b = 1/a # ganho em fitness ao perder
matriz_payoff = np.array([[1, b, a],
                          [a, 1, b],
                          [b, a, 1]])
index_map = {'O': 0, 'Y': 1, 'B': 2}
n_geracoes = 100
n_pop = 100 # número de populações independentes
tipo = "homogenea" # tipo de vizinhança: "homogenea", "I", "II" ou "adaptativa"
prob_mutacao = None # probabilidade de mutação a cada geração
threshold_O = 0.9 # threshold para a proporção de O na vizinhança adaptativa

class Lagarto:
  def __init__(self, i, j, estrategia, fitness, coord_vizinhos, estrategia_vizinhos, coord_vizinhanca_extendida, estrategia_vizinhanca_extendida, tipo_vizinhanca, t, n_vizinhos):
    self.i = i # linha
    self.j = j # coluna
    self.estrategia = estrategia
    self.fitness = 0 # inicia com 0 de fitness
    self.coord_vizinhos = [] # lista vazia para adicionar as coordenadas dos vizinhos
    self.estrategia_vizinhos = [] # lista vazia para adicionar as estratégias dos vizinhos
    self.coord_vizinhanca_extendida = []
    self.estrategia_vizinhanca_extendida = []
    self.tipo_vizinhanca = tipo # tipo de vizinhança "homogenea", "I" ou "II"
    self.t = 0 # determina a geracao do lagarto
    self.n_vizinhos = 0 # número de vizinhos

  def calcular_coord_vizinhos(self, L, matriz_posicao, threshold_O): # obtém as coordenadas dos vizinhos

      if self.tipo_vizinhanca == "homogenea":
        radius_map = {'B': 1, 'O': 1, 'Y': 1} # raio da vizinhança homogenea: todos com vizinhança de Moore (8 vizinhos)
        r = radius_map.get(self.estrategia, 1) # verifica qual a estrategia do lagarto pra saber qual vizinhança usar

        lista_vizinhos = []
        for x in range(-r, r + 1):
            for y in range(-r, r + 1):
                if x == 0 and y == 0: # não inclui o próprio lagarto na lista vizinhos
                    continue
                ni = (self.i + x) % L # fronteiras periódicas (obtém a linha)
                nj = (self.j + y) % L # obtém a coluna
                lista_vizinhos.append((ni, nj)) # retorna duplex (linha, coluna)
        self.coord_vizinhos = lista_vizinhos

      elif self.tipo_vizinhanca == "I":
        radius_map = {'B': 1, 'O': 2, 'Y': 3} # raio da vizinhança Y > O > B
        r = radius_map.get(self.estrategia, 1) # verifica qual a estrategia do lagarto pra saber qual vizinhança usar

        lista_vizinhos = []
        for x in range(-r, r + 1):
            for y in range(-r, r + 1):
                if x == 0 and y == 0: 
                    continue
                ni = (self.i + x) % L 
                nj = (self.j + y) % L 
                lista_vizinhos.append((ni, nj)) 
        self.coord_vizinhos = lista_vizinhos

      elif self.tipo_vizinhanca == "II":
        radius_map = {'B': 2, 'O': 3, 'Y': 2} # raio da vizinhança O > Y = B
        r = radius_map.get(self.estrategia, 1) # verifica qual a estrategia do lagarto pra saber qual vizinhança usar

        lista_vizinhos = []
        for x in range(-r, r + 1):
            for y in range(-r, r + 1):
                if x == 0 and y == 0: 
                    continue
                ni = (self.i + x) % L 
                nj = (self.j + y) % L
                lista_vizinhos.append((ni, nj)) 
        self.coord_vizinhos = lista_vizinhos
      
      elif self.tipo_vizinhanca == "adaptativa":
        radius_map = {'B': 2, 'O': 3, 'Y': 2} # raio da vizinhança adaptativa (Y pode ser igual a B ou O, dependendo do threshold)
        r = radius_map.get(self.estrategia, 1) # verifica qual a estrategia do lagarto pra saber qual vizinhança usar 

        lista_vizinhos = []
        if self.estrategia == 'O' or self.estrategia == "B" : # se for O ou B, usa vizinhança normal
          for x in range(-r, r + 1):
              for y in range(-r, r + 1):
                  if x == 0 and y == 0: 
                    continue 
                  ni = (self.i + x) % L 
                  nj = (self.j + y) % L 
                  lista_vizinhos.append((ni, nj)) 
        
        elif self.estrategia == 'Y': # se for Y, usa vizinhança adaptativa
            for x in range(-r, r + 1): # inicia com r = 2
              for y in range(-r, r + 1):
                  if x == 0 and y == 0: 
                    continue 
                  ni = (self.i + x) % L 
                  nj = (self.j + y) % L 
                  lista_vizinhos.append((ni, nj)) 
            estrategia_vizinhos = [matriz_posicao[ni, nj] for ni, nj in lista_vizinhos] # verifica quem são os vizinhos
            print('Proporção de vizinhos O:', sum(e == 'O' for e in estrategia_vizinhos) / len(estrategia_vizinhos))
            if sum(e == 'O' for e in estrategia_vizinhos) < threshold_O * len(estrategia_vizinhos): # se a proporção de O na vizinhança for menor que threshold, usa vizinhança maior
               r = 3 # vizinhança maior
               lista_vizinhos = []
               for x in range(-r, r + 1):
                 for y in range(-r, r + 1):
                   if x == 0 and y == 0: 
                    continue 
                   ni = (self.i + x) % L 
                   nj = (self.j + y) % L 
                   lista_vizinhos.append((ni, nj)) 

        self.coord_vizinhos = lista_vizinhos
    
  def obter_estrategia_vizinhos(self, matriz_posicao):
      self.estrategia_vizinhos = [matriz_posicao[ni, nj] for ni, nj in self.coord_vizinhos] # dadas as coordenadas, obtém a estratégia do lagarto que ocupa aquela posição

  def mutacao(self, prob_mutacao): # função de mutação
    if np.random.rand() < prob_mutacao: # sorteia um valor entre 0 e 1, se for menor que a probabilidade de mutação, o lagarto muda de estratégia
        estrategias_possiveis = [e for e in estrategias if e != self.estrategia] # obtém as estratégias possíveis, exceto a atual
        self.estrategia = np.random.choice(estrategias_possiveis) # escolhe uma nova estratégia aleatoriamente para mutar

  def calcular_n_vizinhos(self): # calcula o número de vizinhos
      self.n_vizinhos = len(self.estrategia_vizinhos) + len(self.estrategia_vizinhanca_extendida)

def calcular_media_vizinhos(lagartos, estrategias):
    medias = []
    for e in estrategias:
        viz = [lag.n_vizinhos for lag in lagartos if lag.estrategia == e]
        medias.append(np.mean(viz) if len(viz) > 0 else 0)
    return medias # retorna a média de vizinhos para cada estratégia

def ajustar_vizinhos_reciprocos(lagartos): # garante que se A é vizinho de B, B também é vizinho de A, pois as interações são recíprocas
    mapa = {(l.i, l.j): l for l in lagartos} # dicionário pra acessar lagartos pela posição

    for l in lagartos:
        for (ni, nj) in l.coord_vizinhos: # vai em todos os vizinhos do lagarto atual (l)
            vizinho = mapa[(ni, nj)]
            # se o lagarto atual (l) não estiver na lista de vizinhos do vizinho, adiciona em vizinhanca_extendida
            if (l.i, l.j) not in vizinho.coord_vizinhos:
                vizinho.estrategia_vizinhanca_extendida.append(str(l.estrategia))
                vizinho.coord_vizinhanca_extendida.append((l.i, l.j))

In [ ]:
# Teste função calcular_coord_vizinhos()

# --- parâmetros mínimos ---
L = 7
threshold_O = 0.1
tipo = "adaptativa"  # pode mudar para "I", "II" ou "adaptativa"

# --- matriz de estratégias ---
np.random.seed(0)
estrategias = ['O', 'Y', 'B']
matriz_posicao = np.random.choice(estrategias, size=(L, L))

print("Matriz de estratégias:")
print(matriz_posicao)

lag = Lagarto(i=3, j=3,
              estrategia='O',
              fitness=0,
              coord_vizinhos=[],
              estrategia_vizinhos=[],
              coord_vizinhanca_extendida=[],
              estrategia_vizinhanca_extendida=[],
              tipo_vizinhanca=tipo,
              t=0,
              n_vizinhos=0)

# --- chama a função que queremos testar ---
lag.calcular_coord_vizinhos(L=L, matriz_posicao=matriz_posicao, threshold_O=threshold_O)

# --- imprime o resultado ---
print("\nCoordenadas dos vizinhos:")
print(lag.coord_vizinhos)
print(f"\nTotal de vizinhos: {len(lag.coord_vizinhos)}")

Matriz de estratégias:
[['O' 'Y' 'O' 'Y' 'Y' 'B' 'O']
 ['B' 'O' 'O' 'O' 'B' 'Y' 'B']
 ['B' 'O' 'Y' 'Y' 'Y' 'Y' 'O']
 ['Y' 'O' 'O' 'Y' 'B' 'O' 'B']
 ['O' 'Y' 'Y' 'B' 'O' 'Y' 'Y']
 ['Y' 'O' 'B' 'O' 'B' 'B' 'O']
 ['B' 'O' 'O' 'O' 'Y' 'Y' 'B']]

Coordenadas dos vizinhos:
[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (3, 0), (3, 1), (3, 2), (3, 4), (3, 5), (3, 6), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4), (4, 5), (4, 6), (5, 0), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5), (5, 6), (6, 0), (6, 1), (6, 2), (6, 3), (6, 4), (6, 5), (6, 6)]

Total de vizinhos: 48


In [27]:
# Teste função obter_estrategia_vizinhos()

# --- parâmetros mínimos ---
L = 7
threshold_O = 0.9
tipo = "I"

# --- matriz de estratégias ---
np.random.seed(1)
estrategias = ['O', 'Y', 'B']
matriz_posicao = np.random.choice(estrategias, size=(L, L))
print("Matriz de estratégias:")
print(matriz_posicao)

# --- cria um lagarto ---
lag = Lagarto(i=2, j=2,
              estrategia='Y',
              fitness=0,
              coord_vizinhos=[],
              estrategia_vizinhos=[],
              coord_vizinhanca_extendida=[],
              estrategia_vizinhanca_extendida=[],
              tipo_vizinhanca=tipo,
              t=0,
              n_vizinhos=0)

# --- calcula vizinhos ---
lag.calcular_coord_vizinhos(L=L, matriz_posicao=matriz_posicao, threshold_O=threshold_O)

# --- agora testamos obter_estrategia_vizinhos ---
lag.obter_estrategia_vizinhos(matriz_posicao)

# --- mostra resultados ---
print("\nCoordenadas dos vizinhos:")
print(lag.coord_vizinhos)

print("\nEstratégias dos vizinhos (na mesma ordem):")
print(lag.estrategia_vizinhos)

print(f"\nTotal de vizinhos: {len(lag.coord_vizinhos)}")


Matriz de estratégias:
[['Y' 'O' 'O' 'Y' 'Y' 'O' 'O']
 ['Y' 'O' 'Y' 'O' 'B' 'Y' 'B']
 ['O' 'B' 'Y' 'B' 'O' 'O' 'B']
 ['O' 'Y' 'B' 'B' 'O' 'Y' 'Y']
 ['B' 'O' 'B' 'Y' 'Y' 'Y' 'Y']
 ['B' 'Y' 'Y' 'O' 'O' 'Y' 'O']
 ['O' 'Y' 'B' 'Y' 'O' 'B' 'B']]

Coordenadas dos vizinhos:
[(6, 6), (6, 0), (6, 1), (6, 2), (6, 3), (6, 4), (6, 5), (0, 6), (0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (1, 6), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 6), (2, 0), (2, 1), (2, 3), (2, 4), (2, 5), (3, 6), (3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (3, 5), (4, 6), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4), (4, 5), (5, 6), (5, 0), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5)]

Estratégias dos vizinhos (na mesma ordem):
['B', 'O', 'Y', 'B', 'Y', 'O', 'B', 'O', 'Y', 'O', 'O', 'Y', 'Y', 'O', 'B', 'Y', 'O', 'Y', 'O', 'B', 'Y', 'B', 'O', 'B', 'B', 'O', 'O', 'Y', 'O', 'Y', 'B', 'B', 'O', 'Y', 'Y', 'B', 'O', 'B', 'Y', 'Y', 'Y', 'O', 'B', 'Y', 'Y', 'O', 'O', 'Y']

Total de vizinhos: 48


In [ ]:
# Teste função mutacao()

estrategias = ['O', 'Y', 'B']

# cria um lagarto fixo
lag = Lagarto(i=0, j=0,
              estrategia='O',
              fitness=0,
              coord_vizinhos=[],
              estrategia_vizinhos=[],
              coord_vizinhanca_extendida=[],
              estrategia_vizinhanca_extendida=[],
              tipo_vizinhanca="homogenea",
              t=0,
              n_vizinhos=0)

print(f"Estratégia inicial: {lag.estrategia}")

# --- testando sem mutação (probabilidade 0) ---
lag.mutacao(prob_mutacao=0)
print(f"Após mutação (prob=0): {lag.estrategia}")

# --- testando mutação certa (probabilidade 1) ---
print(f"Antes mutação (prob=1): {lag.estrategia}")
lag.mutacao(prob_mutacao=1)
print(f"Após mutação (prob=1): {lag.estrategia}")

# --- testando estatisticamente ---
print("\nTestando estatisticamente com prob_mutacao=0.01")
n_testes = 10000
mudancas = 0
for _ in range(n_testes):
    lag.estrategia = 'O'
    lag.mutacao(prob_mutacao=0.01)
    if lag.estrategia != 'O':
        mudancas += 1
print(f"Ocorreu mutação em {mudancas}/{n_testes} tentativas")


Estratégia inicial: O
Após mutação (prob=0): O
Antes mutação (prob=1): O
Após mutação (prob=1): B

Testando estatisticamente com prob_mutacao=0.5
Ocorreu mutação em 103/10000 tentativas


In [35]:
# Teste função calcular_n_vizinhos()

# --- parâmetros mínimos ---
L = 7
threshold_O = 0.9
tipo = "I"

# --- matriz de estratégias ---
np.random.seed(1)
estrategias = ['O', 'Y', 'B']
matriz_posicao = np.random.choice(estrategias, size=(L, L))
print("Matriz de estratégias:")
print(matriz_posicao)

# --- cria um lagarto ---
lag = Lagarto(i=2, j=2,
              estrategia='B',
              fitness=0,
              coord_vizinhos=[],
              estrategia_vizinhos=[],
              coord_vizinhanca_extendida=[],
              estrategia_vizinhanca_extendida=[],
              tipo_vizinhanca=tipo,
              t=0,
              n_vizinhos=0)

# --- calcula vizinhos ---
lag.calcular_coord_vizinhos(L=L, matriz_posicao=matriz_posicao, threshold_O=threshold_O)

# --- agora testamos obter_estrategia_vizinhos ---
lag.obter_estrategia_vizinhos(matriz_posicao)

# colocando mais vizinhos estendidos manualmente para testar
lag.estrategia_vizinhanca_extendida = ['O', 'B']  # 2 vizinhos extras

# chama a função
lag.calcular_n_vizinhos()

# imprime resultado
print(f"Vizinhos diretos: {len(lag.estrategia_vizinhos)}")
print(f"Vizinhos estendidos: {len(lag.estrategia_vizinhanca_extendida)}")
print(f"Total calculado (n_vizinhos): {lag.n_vizinhos}")

Matriz de estratégias:
[['Y' 'O' 'O' 'Y' 'Y' 'O' 'O']
 ['Y' 'O' 'Y' 'O' 'B' 'Y' 'B']
 ['O' 'B' 'Y' 'B' 'O' 'O' 'B']
 ['O' 'Y' 'B' 'B' 'O' 'Y' 'Y']
 ['B' 'O' 'B' 'Y' 'Y' 'Y' 'Y']
 ['B' 'Y' 'Y' 'O' 'O' 'Y' 'O']
 ['O' 'Y' 'B' 'Y' 'O' 'B' 'B']]
Vizinhos diretos: 8
Vizinhos estendidos: 2
Total calculado (n_vizinhos): 10


In [38]:
# Teste função calcular_media_vizinhos()


# --- cria alguns lagartos com diferentes estratégias e números de vizinhos ---
lagartos = [
    Lagarto(0, 0, 'O', 0, [], [], [], [], "homogenea", 0, 0),
    Lagarto(0, 1, 'O', 0, [], [], [], [], "homogenea", 0, 0),
    Lagarto(1, 0, 'Y', 0, [], [], [], [], "homogenea", 0, 0),
    Lagarto(1, 1, 'B', 0, [], [], [], [], "homogenea", 0, 0),
    Lagarto(2, 2, 'B', 0, [], [], [], [], "homogenea", 0, 0)
]

# atribui valores de n_vizinhos (como se já tivessem sido calculados)
lagartos[0].n_vizinhos = 0   # O
lagartos[1].n_vizinhos = 0  # O
lagartos[2].n_vizinhos = 2  # Y
lagartos[3].n_vizinhos = 7   # B
lagartos[4].n_vizinhos = 6   # B

estrategias = ['O', 'Y', 'B']

# --- testa a função ---
medias = calcular_media_vizinhos(lagartos, estrategias)

# --- mostra resultados ---
for e, m in zip(estrategias, medias):
    print(f"Estratégia {e}: média de vizinhos = {m}")


Estratégia O: média de vizinhos = 0.0
Estratégia Y: média de vizinhos = 2.0
Estratégia B: média de vizinhos = 6.5


In [ ]:
L = 7
matriz_posicao = np.full((L, L), 'O')  # preenche tudo com 'Y'
centro = L // 2
matriz_posicao[centro, centro] = 'B'   # coloca o 'B' no meio
print(matriz_posicao)

lagartos = []

for i in range(L):
    for j in range(L):
        lag = Lagarto(
            i=i,
            j=j,
            estrategia=matriz_posicao[i, j],
            fitness=0,
            coord_vizinhos=[],
            estrategia_vizinhos=[],
            coord_vizinhanca_extendida=[],
            estrategia_vizinhanca_extendida=[],
            tipo_vizinhanca="I",
            t=0,
            n_vizinhos=0
        )
        lagartos.append(lag)

for lag in lagartos:
    lag.calcular_coord_vizinhos(L, matriz_posicao, threshold_O)
    lag.obter_estrategia_vizinhos(matriz_posicao)

# --- chama a função a testar ---
ajustar_vizinhos_reciprocos(lagartos)

for lag in lagartos:
    lag.calcular_n_vizinhos()

# cria um dicionário com chave (i,j) e valor Lagarto
mapa_lagartos = {(lag.i, lag.j): lag for lag in lagartos}

# agora o acesso é direto
lag_centro = mapa_lagartos[(3, 3)]
print(lag_centro.estrategia)
print(lag_centro.coord_vizinhos)
print(lag_centro.estrategia_vizinhos)
print(lag_centro.coord_vizinhanca_extendida)
print(lag_centro.estrategia_vizinhanca_extendida)
print(lag_centro.n_vizinhos)

[['O' 'O' 'O' 'O' 'O' 'O' 'O']
 ['O' 'O' 'O' 'O' 'O' 'O' 'O']
 ['O' 'O' 'O' 'O' 'O' 'O' 'O']
 ['O' 'O' 'O' 'B' 'O' 'O' 'O']
 ['O' 'O' 'O' 'O' 'O' 'O' 'O']
 ['O' 'O' 'O' 'O' 'O' 'O' 'O']
 ['O' 'O' 'O' 'O' 'O' 'O' 'O']]
B
[(2, 2), (2, 3), (2, 4), (3, 2), (3, 4), (4, 2), (4, 3), (4, 4)]
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
[(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 1), (2, 5), (3, 1), (3, 5), (4, 1), (4, 5), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5)]
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
24
